In [18]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter


In [19]:

# Dataset class
class CloudDataset(Dataset):
    def __init__(self, base_folder, wind_data_folder, resize=(128, 128)):
        self.inputs = []
        self.labels = []
        self.resize = resize
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(resize)
        ])

        for subfolder in os.listdir(base_folder):
            subfolder_path = os.path.join(base_folder, subfolder)
            if not os.path.isdir(subfolder_path) or not subfolder.startswith("Day_"):
                continue

            day_number = int(subfolder.split("_")[1])
            wind_filename = f"E06SCTL4AW_2024{day_number:03}_25km_v1.0.2.npy"
            wind_file_path = os.path.join(wind_data_folder, wind_filename)

            if not os.path.exists(wind_file_path):
                continue

            wind_data = np.load(wind_file_path)
            images = sorted([f for f in os.listdir(subfolder_path) if f.startswith("mask_")])

            for i in range(len(images) - 2):
                img1 = plt.imread(os.path.join(subfolder_path, images[i]))
                img2 = plt.imread(os.path.join(subfolder_path, images[i + 2]))
                intermediate = plt.imread(os.path.join(subfolder_path, images[i + 1]))

                img1 = self.transform(img1)
                img2 = self.transform(img2)
                intermediate = self.transform(intermediate)

                wind_data_flat = torch.tensor(wind_data.mean(axis=(0, 1)), dtype=torch.float32)
                wind_data_expanded = wind_data_flat.unsqueeze(-1).unsqueeze(-1).expand(3, *img1.shape[1:])
                input_tensor = torch.cat((img1, img2, wind_data_expanded), dim=0)

                self.inputs.append(input_tensor)
                self.labels.append(intermediate)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]


In [20]:

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
base_folder = "V3/mapped_output"
wind_data_folder = "V2/wind/wind_arrays"
# ConvLSTMCell


Using device: cuda


In [21]:

# ConvLSTM Model Code (unchanged)
class ConvLSTMCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size):
        super(ConvLSTMCell, self).__init__()
        self.input_channels = input_channels
        self.hidden_channels = hidden_channels
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels=input_channels + hidden_channels,
            out_channels=4 * hidden_channels,  # for input, forget, cell, and output gates
            kernel_size=kernel_size,
            padding=self.padding
        )

    def forward(self, x, hidden_state):
        h_prev, c_prev = hidden_state
        combined = torch.cat([x, h_prev], dim=1)  # Concatenate input and previous hidden state
        gates = self.conv(combined)
        input_gate, forget_gate, cell_gate, output_gate = torch.chunk(gates, chunks=4, dim=1)

        input_gate = torch.sigmoid(input_gate)
        forget_gate = torch.sigmoid(forget_gate)
        output_gate = torch.sigmoid(output_gate)
        cell_gate = torch.tanh(cell_gate)

        c_next = forget_gate * c_prev + input_gate * cell_gate
        h_next = output_gate * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(self, batch_size, height, width):
        return (torch.zeros(batch_size, self.hidden_channels, height, width).to(next(self.parameters()).device),
                torch.zeros(batch_size, self.hidden_channels, height, width).to(next(self.parameters()).device))


In [22]:

class ConvLSTM(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size, num_layers):
        super(ConvLSTM, self).__init__()
        self.num_layers = num_layers
        self.hidden_channels = hidden_channels

        self.cells = nn.ModuleList([
            ConvLSTMCell(
                input_channels=input_channels if i == 0 else hidden_channels,
                hidden_channels=hidden_channels,
                kernel_size=kernel_size
            ) for i in range(num_layers)
        ])

    def forward(self, x, hidden_states=None):
        batch_size, _, height, width = x.size()
        if hidden_states is None:
            hidden_states = [cell.init_hidden(batch_size, height, width) for cell in self.cells]

        current_input = x
        new_hidden_states = []
        for i, cell in enumerate(self.cells):
            h_next, c_next = cell(current_input, hidden_states[i])
            new_hidden_states.append((h_next, c_next))
            current_input = h_next

        return current_input, new_hidden_states


In [23]:

# Frame Generator
class FrameGenerator(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size, num_layers):
        super(FrameGenerator, self).__init__()
        self.encoder1 = nn.Conv2d(input_channels, hidden_channels, kernel_size, padding=kernel_size // 2)
        self.encoder2 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size, padding=kernel_size // 2)
        self.lstm = ConvLSTM(hidden_channels, hidden_channels, kernel_size, num_layers)
        self.decoder1 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size, padding=kernel_size // 2)
        self.decoder2 = nn.Conv2d(hidden_channels, 3, kernel_size, padding=kernel_size // 2)

    def forward(self, x):
        x = torch.relu(self.encoder1(x))
        x = torch.relu(self.encoder2(x))
        lstm_output, _ = self.lstm(x)
        lstm_output = torch.sigmoid(self.decoder1(lstm_output))
        return torch.sigmoid(self.decoder2(lstm_output))


In [24]:

# Model, Loss, Optimizer
input_channels = 9
hidden_channels = 64
kernel_size = 3
num_layers = 2

model = FrameGenerator(input_channels, hidden_channels, kernel_size, num_layers).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# Load Dataset

dataset = CloudDataset(base_folder, wind_data_folder)


In [25]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)



In [ ]:

# # Model, Loss, Optimizer
# input_channels = 9
# hidden_channels = 64
# kernel_size = 3
# num_layers = 2

# model = FrameGenerator(input_channels, hidden_channels, kernel_size, num_layers).to(device)
# criterion = nn.MSELoss()
# optimizer = optim.AdamW(model.parameters(), lr=0.001)



In [37]:

# Training
writer = SummaryWriter()
epochs = 100

for epoch in range(epochs):
    print(f"Epoch {epoch+1}")
    torch.cuda.empty_cache()
    model.train()
    train_loss = 0.0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        print(batch_idx, end='\r')
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, targets).item()
    val_loss /= len(test_loader)

    writer.add_scalars("Loss", {"Train": train_loss, "Validation": val_loss}, epoch)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

writer.close()
torch.save(model.state_dict(), "conv_lstm_model.pth")

Epoch 1
Epoch 1/100, Train Loss: 0.0433, Validation Loss: 0.0146
Epoch 2
Epoch 2/100, Train Loss: 0.0109, Validation Loss: 0.0080
Epoch 3
Epoch 3/100, Train Loss: 0.0074, Validation Loss: 0.0069
Epoch 4
Epoch 4/100, Train Loss: 0.0066, Validation Loss: 0.0062
Epoch 5
Epoch 5/100, Train Loss: 0.0061, Validation Loss: 0.0065
Epoch 6
Epoch 6/100, Train Loss: 0.0059, Validation Loss: 0.0058
Epoch 7
Epoch 7/100, Train Loss: 0.0057, Validation Loss: 0.0060
Epoch 8
Epoch 8/100, Train Loss: 0.0056, Validation Loss: 0.0056
Epoch 9
Epoch 9/100, Train Loss: 0.0056, Validation Loss: 0.0056
Epoch 10
Epoch 10/100, Train Loss: 0.0055, Validation Loss: 0.0056
Epoch 11
Epoch 11/100, Train Loss: 0.0054, Validation Loss: 0.0054
Epoch 12
Epoch 12/100, Train Loss: 0.0053, Validation Loss: 0.0054
Epoch 13
Epoch 13/100, Train Loss: 0.0054, Validation Loss: 0.0055
Epoch 14
Epoch 14/100, Train Loss: 0.0053, Validation Loss: 0.0054
Epoch 15
Epoch 15/100, Train Loss: 0.0053, Validation Loss: 0.0052
Epoch 16
Epoc

KeyboardInterrupt: 

In [41]:
torch.save(model.state_dict(), "conv_lstm_model_09_12.pth")

In [26]:
import os
import cv2
import numpy as np

output_folder = "video_gen"
os.makedirs(output_folder, exist_ok=True)

def save_images_to_video(final_images, output_path, fps=5):
    """
    Save a sequence of images as a video.
    
    Args:
        final_images: List of images (tensors) to save as video.
        output_path: Path to save the output video file.
        fps: Frames per second for the video.
    """
    # Convert tensor images to NumPy arrays and to OpenCV format
    processed_images = []
    for img_tensor in final_images:
        if img_tensor is not None:
            # Convert tensor (3, H, W) to NumPy array (H, W, 3)
            img_np = (img_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
            processed_images.append(cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
    
    if not processed_images:
        print("No images to save in the video.")
        return

    # Get video resolution
    height, width, _ = processed_images[0].shape

    # Define video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for MP4 format
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Write frames to the video
    for img in processed_images:
        video_writer.write(img)

    video_writer.release()
    print(f"Video saved at: {output_path}")


In [ ]:
def generate_images(model, images, wind_data, left_idx, right_idx, results):
    """
    Recursively generate intermediate images using divide-and-conquer.
    Args:
        model: The trained model.
        images: Tensor of input images with shape (N, 3, H, W).
        wind_data: Tensor of wind data corresponding to images (N, 3, H, W).
        left_idx: Leftmost image index in the range.
        right_idx: Rightmost image index in the range.
        results: Dictionary storing generated images by index.
    """
    # Base case: If the indices are consecutive, return
    if left_idx + 1 >= right_idx:
        return

    # Get the left and right images
    img_left = images[left_idx]
    img_right = images[right_idx]

    # Check for None values in images
    if img_left is None or img_right is None:
        raise ValueError(f"Image data at indices {left_idx} or {right_idx} is None.")

    # Combine wind data from the leftmost and rightmost images
    wind_left = wind_data[left_idx]
    wind_right = wind_data[right_idx]

    # Check for None values in wind data
    if wind_left is None or wind_right is None:
        raise ValueError(f"Wind data at indices {left_idx} or {right_idx} is None.")

    combined_wind = (wind_left + wind_right) / 2  # Average the wind channels

    # Prepare input tensor for the model
    input_tensor = torch.cat((img_left, img_right, combined_wind), dim=0).unsqueeze(0)  # Add batch dimension

    # Pass through the model to generate the middle image
    with torch.no_grad():
        middle_image = model(input_tensor).squeeze(0)  # Remove batch dimension

    # Find the middle index
    middle_idx = (left_idx + right_idx) // 2

    # Store the generated middle image and interpolate wind data
    images[middle_idx] = middle_image  # Fill images list with the generated image
    wind_data[middle_idx] = combined_wind  # Fill wind data for the middle index
    results[middle_idx] = middle_image

    # Recursively generate images for left and right halves
    generate_images(model, images, wind_data, left_idx, middle_idx, results)
    generate_images(model, images, wind_data, middle_idx, right_idx, results)


# model = DummyModel()
model.load_state_dict(torch.load("conv_lstm_model_09_12.pth", weights_only=True))
model.eval()
#  Main loop: Save generated videos
max_sets = 5
current_set = 0

for batch_idx, (inputs, targets) in enumerate(test_loader):
    batch_size = inputs.shape[0]
    for b in range(batch_size):
        if current_set >= max_sets:
            break

        # Prepare input data
        inputs = inputs.to(device)
        img1 = inputs[b, :3, :, :]
        img2 = inputs[b, 3:6, :, :]
        wind = inputs[b, 6:, :, :]

        num_images = 24
        images = [None] * num_images
        wind_data = [None] * num_images

        images[0] = img1
        images[-1] = img2
        wind_data[0] = wind
        wind_data[-1] = wind

        results = {0: img1, num_images - 1: img2}

        # Generate images recursively
        generate_images(model, images, wind_data, 0, num_images - 1, results)

        # Final sequence of images
        final_images = [results[i] if i in results else None for i in range(num_images)]

        # Save images as a video
        video_name = f"generated_video_{current_set + 1}.mp4"
        output_path = os.path.join(output_folder, video_name)
        save_images_to_video(final_images, output_path, fps=5)

        current_set += 1

    if current_set >= max_sets:
        break


Video saved at: video_gen\generated_video_1.mp4
Video saved at: video_gen\generated_video_2.mp4
Video saved at: video_gen\generated_video_3.mp4
Video saved at: video_gen\generated_video_4.mp4
Video saved at: video_gen\generated_video_5.mp4


In [58]:
sum(p.numel() for p in model.parameters() if p.requires_grad)


671171

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

output_folder = "video_gen_long"
os.makedirs(output_folder, exist_ok=True)

# Save images as a video
def save_images_to_video(final_images, output_path, fps=5):
    processed_images = []
    for img_tensor in final_images:
        if img_tensor is not None:
            img_np = (img_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
            processed_images.append(cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

    if not processed_images:
        print("No images to save in the video.")
        return

    height, width, _ = processed_images[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for img in processed_images:
        video_writer.write(img)

    video_writer.release()
    print(f"Video saved at: {output_path}")

# New Dataset class
class NewCloudDataset(Dataset):
    def __init__(self, image_folder, wind_folder, resize=(128, 128)):
        self.data = []
        self.resize = resize
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(resize)
        ])

        for day_folder in sorted(os.listdir(image_folder)):
            day_image_path = os.path.join(image_folder, day_folder)
            day_wind_path = os.path.join(wind_folder, day_folder)

            if not os.path.isdir(day_image_path) or not os.path.isdir(day_wind_path):
                continue

            wind_files = sorted([f for f in os.listdir(day_wind_path) if f.endswith(".npy")])
            image_files = sorted([f for f in os.listdir(day_image_path) if f.startswith("mask_")])

            if len(image_files) < 2 or len(wind_files) == 0:
                continue

            wind_data = np.load(os.path.join(day_wind_path, wind_files[0]))
            wind_tensor = torch.tensor(wind_data.mean(axis=(0, 1)), dtype=torch.float32)

            for i in range(len(image_files) - 1):
                img1 = plt.imread(os.path.join(day_image_path, image_files[i]))
                img2 = plt.imread(os.path.join(day_image_path, image_files[i + 1]))

                img1 = self.transform(img1)
                img2 = self.transform(img2)

                wind_expanded = wind_tensor.unsqueeze(-1).unsqueeze(-1).expand(3, *img1.shape[1:])
                input_tensor = torch.cat((img1, img2, wind_expanded), dim=0)

                self.data.append(input_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Recursive image generation with memoization
def generate_images(model, left_image, right_image, left_wind, right_wind, depth):
    """
    Recursively generate intermediate frames between left_image and right_image.

    Parameters:
    - model: The frame interpolation model.
    - left_image: The starting image tensor (3 x H x W).
    - right_image: The ending image tensor (3 x H x W).
    - left_wind: Wind tensor for the left image (3 x 1 x 1).
    - right_wind: Wind tensor for the right image (3 x 1 x 1).
    - depth: The remaining depth of recursion.

    Returns:
    - A list of intermediate frames in correct temporal order.
    """
    if depth == 0:
        return []

    # Compute the average wind for the midpoint
    mid_wind = (left_wind + right_wind) / 2

    # Prepare input tensor for the model
    input_tensor = torch.cat((left_image, right_image, mid_wind.expand_as(left_image)), dim=0).unsqueeze(0)

    # Predict the intermediate frame
    with torch.no_grad():
        mid_frame = model(input_tensor).squeeze(0)  # Output shape: (3 x H x W)

    # Recursively generate frames for the left and right intervals
    left_frames = generate_images(model, left_image, mid_frame, left_wind, mid_wind, depth - 1)
    right_frames = generate_images(model, mid_frame, right_image, mid_wind, right_wind, depth - 1)

    # Combine frames in correct temporal order
    return left_frames + [mid_frame] + right_frames



# Model loading
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = ConvLSTM()  # Replace with actual model class
model.load_state_dict(torch.load("conv_lstm_model_09_12.pth"))
model.to(device)
model.eval()

image_folder = "dataset_vid/PICS"
wind_folder = "dataset_vid/WIND"

# Initialize dataset and dataloader
dataset = NewCloudDataset(image_folder, wind_folder)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

print(f"Total images loaded: {len(dataset)}")

# Generate extended video
all_frames = []

recursion_depth = 5  # Determines 2^5 = 32 frames between each pair

for batch_idx, input_tensor in enumerate(dataloader):
    input_tensor = input_tensor[0].to(device)

    left_image = input_tensor[:3]
    right_image = input_tensor[3:6]
    left_wind = input_tensor[6:].mean(dim=(1, 2), keepdim=True)
    right_wind = input_tensor[6:].mean(dim=(1, 2), keepdim=True)

    all_frames.append(left_image)

    interpolated_frames = generate_images(model, left_image, right_image, left_wind, right_wind, recursion_depth)
    all_frames.extend(interpolated_frames)

    all_frames.append(right_image)

video_name = "complete_extended_video.mp4"
output_path = os.path.join(output_folder, video_name)
save_images_to_video(all_frames, output_path, fps=10)

print("Extended video generation complete.")

C:\Users\Sahil\AppData\Local\Temp\ipykernel_36364\3938239393.py:119: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("conv_lstm_model_09_12.pt

Total images loaded: 198
Video saved at: video_gen_long\complete_extended_video.mp4
Extended video generation complete.
